<a href="https://colab.research.google.com/github/LampOfSocrates/Surrey_EEEM071_Coursework/blob/main/EEEM071_CourseWork_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EEEM071 — Vehicle Re-ID Training

Runs in **Google Colab** (mounts Drive, clones repo, installs deps) and **locally** (uses repo working directory). Environment is detected automatically via `in_colab()`.

Parameterised with [Papermill](https://papermill.readthedocs.io). Pre-configured experiment notebooks live in `notebooks/`. Run the full LR sweep with:
```bash
python scripts/run_lr_sweep.py
```

In [1]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# "single"          → train / eval one experiment (uses parameters below)
# "lr_sweep"        → MobileNetV3-Small LR sweep (5 runs) + comparison
# "all_experiments" → generate & run all 9 experiment notebooks + comparison
# "compare"         → compare existing results only (no training)
run_mode = "lr_sweep"

# ── Student info ──────────────────────────────────────────────────────────────
student_id   = "your_student_id"
student_name = "Your_Name"

# ── Dataset ───────────────────────────────────────────────────────────────────
source_dataset = "veri"         # dataset used for training
target_dataset = "veri"         # dataset used for evaluation
data_root      = "../data/VeRi" # local path; overridden automatically in Colab

# ── Colab settings (ignored when running locally) ─────────────────────────────
colab_repo_url        = "https://github.com/LampOfSocrates/Surrey_EEEM071_Coursework.git"
colab_repo_dir        = "/content/Surrey_EEEM071_Coursework"
colab_drive_data_path = "/content/drive/MyDrive"

# ── Model ─────────────────────────────────────────────────────────────────────
arch          = "mobilenet_v3_small"  # resnet50 | resnet18 | mobilenet_v3_small | vgg16
no_pretrained = False                 # set True to skip ImageNet pretrained weights

# ── Image dimensions ──────────────────────────────────────────────────────────
image_height = 224
image_width  = 224

# ── Optimisation ──────────────────────────────────────────────────────────────
optimizer     = "amsgrad"  # adam | amsgrad | sgd | rmsprop
learning_rate = 0.0003
weight_decay  = 5e-4
max_epochs    = 1
stepsize      = "20 40"   # space-separated LR decay milestones
gamma         = 0.1       # LR decay factor
seed          = 1

# ── Batch sizes ───────────────────────────────────────────────────────────────
train_batch_size = 64
test_batch_size  = 100

# ── Loss ──────────────────────────────────────────────────────────────────────
margin      = 0.3  # triplet loss margin
lambda_xent = 1.0  # cross-entropy loss weight
lambda_htri = 1.0  # hard-triplet loss weight
label_smooth = False

# ── Augmentation ──────────────────────────────────────────────────────────────
random_erase = False
color_jitter = False
color_aug    = False

# ── Hardware & output ─────────────────────────────────────────────────────────
gpu_devices   = "0"                      # e.g. "0" or "0,1"
use_cpu       = False
eval_freq     = -1                       # -1 = evaluate only at end
evaluate_only = False                    # skip training, run eval only
save_dir      = "logs/mobilenet_v3_small-veri-lr3e-4"
data_fraction = 0.01                     # 0.01 = 1% for fast iteration; 1.0 = full dataset

## 1. Environment Detection

In [2]:
import json
import os
import re
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def in_colab() -> bool:
    """Return True when running inside Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IS_COLAB = in_colab()
print(f"Environment : {'Google Colab' if IS_COLAB else 'local'}")
print(f"Python      : {sys.executable}")

Environment : Google Colab
Python      : /usr/bin/python3


## 2a. Colab Setup

> **Colab checklist before running:**
> 1. *Edit → Notebook settings* → set Hardware accelerator to **GPU**
> 2. Click the folder icon on the left and **mount Google Drive** (or wait for the cell below to prompt you)
> 3. Upload `VeRi.zip` to `My Drive/` and note the path — set `colab_drive_data_path` in the parameters cell
>
> *Skipped automatically when not in Colab.*

In [3]:
if IS_COLAB:
    # ── Fallback defaults (used if parameters cell hasn't run yet) ────────────
    colab_repo_url        = globals().get("colab_repo_url",        "https://github.com/LampOfSocrates/Surrey_EEEM071_Coursework.git")
    colab_repo_dir        = globals().get("colab_repo_dir",        "/content/Surrey_EEEM071_Coursework")
    colab_drive_data_path = globals().get("colab_drive_data_path", "/content/drive/MyDrive")

    # ── Step 1: GPU check ────────────────────────────────────────────────────
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("WARNING: nvidia-smi not available. Make sure GPU is enabled in Colab settings.")

    # ── Step 2: Mount Google Drive ───────────────────────────────────────────
    from google.colab import drive
    drive.mount("/content/drive")

    # ── Step 3: Clone / update repo ──────────────────────────────────────────
    if not Path(colab_repo_dir).exists():
        subprocess.run(["git", "clone", colab_repo_url, colab_repo_dir], check=True)
        print(f"Cloned repo -> {colab_repo_dir}")
    else:
        # Force-sync to origin/main — discards any local changes in the runtime
        subprocess.run(["git", "fetch", "origin"], cwd=colab_repo_dir, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=colab_repo_dir, check=True)
        print("Repo updated to latest origin/main")

    os.chdir(colab_repo_dir)
    print(f"Working directory: {Path.cwd()}")

    # ── Step 4: Install gdown ────────────────────────────────────────────────
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", "gdown", "--pre"],
        check=True,
    )

    # ── Step 5: Data note ────────────────────────────────────────────────────
    # Upload VeRi dataset to Google Drive, then set colab_drive_data_path.
    if Path("/content/VeRi.zip").exists():
        print("VeRi.zip found in /content — unzip manually if not already done.")
    print(f"Data root (Colab): {colab_drive_data_path}")

    # Override data_root for all subsequent cells
    data_root = colab_drive_data_path

else:
    print("Colab setup skipped (local environment)")

Tue Mar 24 10:31:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2b. Local Setup

*Skipped automatically when running in Colab.*

In [4]:
if not IS_COLAB:
    # Normalise working directory to the folder containing this notebook
    # so that main.py and src/ are importable regardless of where papermill
    # was invoked from.
    notebook_dir = Path(globals().get("__vsc_ipynb_file__", "")).parent
    if notebook_dir.exists() and notebook_dir != Path("."):
        os.chdir(notebook_dir)
    # Fallback: if this file is in notebooks/, step up one level to repo root
    if not Path("main.py").exists() and Path("../main.py").exists():
        os.chdir("..")
    print(f"Working directory: {Path.cwd()}")

    # GPU check (non-fatal)
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        if result.returncode == 0:
            print("\n".join(result.stdout.splitlines()[:10]))
        else:
            print("nvidia-smi not available — training will use CPU")
    except FileNotFoundError:
        print("nvidia-smi not found — training will use CPU")

else:
    print("Local setup skipped (Colab environment)")

print(f"\nStudent  : {student_name} ({student_id})")
print(f"Arch     : {arch}")
print(f"Dataset  : {source_dataset} -> {target_dataset}")
print(f"Data     : {data_root}")
print(f"Epochs   : {max_epochs}  |  LR: {learning_rate}  |  Optim: {optimizer}")

Local setup skipped (Colab environment)

Student  : Your_Name (your_student_id)
Arch     : mobilenet_v3_small
Dataset  : veri -> veri
Data     : /content/drive/MyDrive
Epochs   : 1  |  LR: 0.0003  |  Optim: amsgrad


## 3. Build Training Command

In [5]:
cmd = [
    sys.executable, "main.py",
    "-s", source_dataset,
    "-t", target_dataset,
    "-a", arch,
    "--root",             data_root,
    "--height",           str(image_height),
    "--width",            str(image_width),
    "--optim",            optimizer,
    "--lr",               str(learning_rate),
    "--weight-decay",     str(weight_decay),
    "--max-epoch",        str(max_epochs),
    "--stepsize",         *stepsize.split(),
    "--gamma",            str(gamma),
    "--train-batch-size", str(train_batch_size),
    "--test-batch-size",  str(test_batch_size),
    "--margin",           str(margin),
    "--lambda-xent",      str(lambda_xent),
    "--lambda-htri",      str(lambda_htri),
    "--seed",             str(seed),
    "--eval-freq",        str(eval_freq),
    "--save-dir",         save_dir,
    "--gpu-devices",      gpu_devices,
]

# Boolean flags
if label_smooth:  cmd.append("--label-smooth")
if random_erase:  cmd.append("--random-erase")
if color_jitter:  cmd.append("--color-jitter")
if color_aug:     cmd.append("--color-aug")
if no_pretrained: cmd.append("--no-pretrained")
if use_cpu:       cmd.append("--use-cpu")
if evaluate_only: cmd.append("--evaluate")

# Inject student credentials via environment variables
env = os.environ.copy()
env["STUDENT_ID"]   = student_id
env["STUDENT_NAME"] = student_name

# Print shell equivalent for reproducibility
shell_cmd = (
    f"STUDENT_ID='{student_id}' STUDENT_NAME='{student_name}' "
    + " ".join(cmd)
)
print(f"Shell equivalent:\n{shell_cmd}")

Shell equivalent:
STUDENT_ID='your_student_id' STUDENT_NAME='Your_Name' /usr/bin/python3 main.py -s veri -t veri -a mobilenet_v3_small --root /content/drive/MyDrive --height 224 --width 224 --optim amsgrad --lr 0.0003 --weight-decay 0.0005 --max-epoch 1 --stepsize 20 40 --gamma 0.1 --train-batch-size 64 --test-batch-size 100 --margin 0.3 --lambda-xent 1.0 --lambda-htri 1.0 --seed 1 --eval-freq -1 --save-dir logs/mobilenet_v3_small-veri-lr3e-4 --gpu-devices 0


## 4. Run Training

In [6]:
if run_mode != "single":
    print(f"Skipping single-run training (run_mode={run_mode!r}). See sections 6–8 below.")
else:
    t0 = time.time()
    _output_lines = []  # captured for metric parsing in the next cell

    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            _output_lines.append(line)
        proc.wait()

    elapsed = time.time() - t0
    print(f"\nTraining finished in {elapsed:.1f}s  (exit code {proc.returncode})")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1

## 5. Metrics Visualisation

Parse training logs and evaluation results, then plot training curves and CMC metrics.

In [7]:
if run_mode != "single":
    print(f"Skipping single-run metrics (run_mode={run_mode!r}).")
else:
    from src.utils.metrics_viz import render_metrics

    _cfg = {
        "arch": arch, "learning_rate": learning_rate, "max_epochs": max_epochs,
        "optimizer": optimizer, "train_batch_size": train_batch_size,
        "image_height": image_height, "image_width": image_width,
        "label_smooth": label_smooth, "random_erase": random_erase,
        "color_jitter": color_jitter, "margin": margin, "save_dir": save_dir,
        "evaluate_only": evaluate_only,
    }
    _metrics = render_metrics(_output_lines, _cfg, elapsed=elapsed)

Parsed 3 batch log entries
mAP       : 10.3%
Rank-1    : 0.0%
Rank-5    : 33.3%
Rank-10   : 33.3%
Rank-20   : 77.8%
Saved: logs/mobilenet_v3_small-veri-lr3e-4/training_curves.png
Saved: logs/mobilenet_v3_small-veri-lr3e-4/eval_metrics.png

Metrics JSON -> logs/mobilenet_v3_small-veri-lr3e-4/metrics.json


## 6. Run LR Sweep

Runs all 5 MobileNetV3-Small learning-rate variants (3e-3 → 3e-5) via Papermill,
exports each to HTML, then runs the comparison notebook.

> Set `run_mode = "lr_sweep"` in the parameters cell to activate this section.

In [8]:
if run_mode != "lr_sweep":
    print(f"Skipping LR sweep (run_mode={run_mode!r}). Set run_mode='lr_sweep' to run.")
else:
    import subprocess, sys
    print("=" * 60)
    print("  Running MobileNetV3-Small LR sweep (5 experiments)")
    print("=" * 60)

    # Generate the 5 LR-sweep notebooks from the master
    subprocess.run([sys.executable, "scripts/generate_experiment_notebooks.py"], check=True)

    # Run all 5 + comparison notebook, export HTML
    result = subprocess.run(
        [sys.executable, "scripts/run_lr_sweep.py"],
        capture_output=False,   # stream to notebook output
    )
    if result.returncode != 0:
        print(f"\nWARNING: run_lr_sweep.py exited with code {result.returncode}")
    else:
        print("\nLR sweep complete.")

Skipping LR sweep (run_mode='single'). Set run_mode='lr_sweep' to run.


## 7. Run ALL Experiments

Generates all 9 experiment notebooks (LR sweep + ResNet-50 + ResNet-18 + SGD/smooth + eval-only)
and runs each one via Papermill sequentially.

> Set `run_mode = "all_experiments"` in the parameters cell to activate this section.

In [9]:
if run_mode != "all_experiments":
    print(f"Skipping all-experiments run (run_mode={run_mode!r}). Set run_mode='all_experiments' to run.")
else:
    import subprocess, sys
    from pathlib import Path

    print("=" * 60)
    print("  Generating all experiment notebooks")
    print("=" * 60)
    subprocess.run([sys.executable, "scripts/generate_experiment_notebooks.py"], check=True)

    nb_dir      = Path("notebooks")
    executed_dir = Path("outputs/executed")
    html_dir     = Path("outputs/html")
    executed_dir.mkdir(parents=True, exist_ok=True)
    html_dir.mkdir(parents=True, exist_ok=True)

    notebooks = sorted(nb_dir.glob("*.ipynb"))
    print(f"\nFound {len(notebooks)} notebooks to run:\n")
    for nb in notebooks:
        print(f"  {nb.name}")

    completed, failed = [], []
    for nb in notebooks:
        out_nb   = executed_dir / nb.name
        out_html = html_dir / nb.with_suffix(".html").name
        print(f"\n{'='*60}\n  Running: {nb.stem}\n{'='*60}")
        try:
            subprocess.run(["papermill", str(nb), str(out_nb)], check=True)
            subprocess.run(
                ["jupyter", "nbconvert", "--to", "html", str(out_nb), "--output", str(out_html)],
                check=True,
            )
            completed.append(nb.stem)
        except subprocess.CalledProcessError as e:
            print(f"  ERROR: {nb.stem} failed (exit {e.returncode}) — continuing")
            failed.append(nb.stem)

    print(f"\n{'='*60}")
    print(f"  Completed : {len(completed)}")
    print(f"  Failed    : {len(failed)}")
    if failed:
        print(f"  Failed    : {', '.join(failed)}")
    print(f"{'='*60}")

Skipping all-experiments run (run_mode='single'). Set run_mode='all_experiments' to run.


## 8. Compare Results

Scans all completed runs under `logs/`, aggregates metrics, and produces a leaderboard
table (CSV / JSON / Markdown) plus a metric glossary explaining each score.

> Runs automatically after `lr_sweep` and `all_experiments` modes.
> Set `run_mode = "compare"` to run standalone against existing logs.

In [14]:
run_mode = 'compare'
if run_mode not in ("compare", "lr_sweep", "all_experiments"):
    print(f"Skipping comparison (run_mode={run_mode!r}). Set run_mode='compare' to run standalone.")
else:
    import subprocess, sys
    print("=" * 60)
    print("  Comparing all completed experiments")
    print("=" * 60)

    result = subprocess.run(
        [sys.executable, "tools/summarize_experiments.py",
         "--logs-dir", "logs",
         "--out-dir",  "outputs/comparison",
         "--top",      "10"],
        capture_output=False,
    )
    if result.returncode == 0:
        from pathlib import Path
        md = Path("outputs/comparison/experiments_summary.md")
        if md.exists():
            print("\n--- experiments_summary.md ---")
            print(md.read_text(encoding="utf-8"))
    else:
        print(f"WARNING: summarize_experiments.py exited with code {result.returncode}")

  Comparing all completed experiments

--- experiments_summary.md ---
# Experiment Summary

Total runs: 1

| Rank | Run dir | Arch | Optim | LR | Epochs | Data% | Best mAP | Best R1 | Final mINP | Time(s) |
|------|---------|------|-------|----|--------|-------|---------|---------|------------|--------|
| 1 | `mobilenet_v3_small-veri-lr3e-4` | mobilenet_v3_small | amsgrad | 0.0003 | 1 | 1% | 10.27% | 0.00% | 9.67% | 70 |

---

## Metric Glossary

**Mean Average Precision (mAP) (`mAP`)** — Area under the precision-recall curve, averaged over all query vehicles. For each query, AP measures how highly the correct matches are ranked. mAP = mean(AP) across all queries. Range: 0-100%. Higher is better.

**Rank-1 Accuracy (`Rank-1`)** — Fraction of queries whose single top-1 retrieved image belongs to the correct vehicle identity. Range: 0-100%. Higher is better.

**Rank-5 Accuracy (`Rank-5`)** — Fraction of queries with at least one correct match in the top-5 results. Range: 0-100%. Higher i

## 6. Output Files

Checkpoints, logs, plots and `metrics.json` are all written to `save_dir`.
The HTML export is generated by `scripts/run_lr_sweep.py` after Papermill finishes.

In [15]:
log_dir = Path(save_dir)
if log_dir.exists():
    files = sorted(log_dir.rglob("*"))
    print(f"Output files in {log_dir}:")
    for f in files:
        if f.is_file():
            print(f"  {str(f.relative_to(log_dir)):<40}  {f.stat().st_size:>10,} bytes")
else:
    print(f"Log directory not yet created: {log_dir}")

print(f"\nRun summary:")
print(f"  arch={arch}, lr={learning_rate}, epochs={max_epochs}, optim={optimizer}, seed={seed}")
print(f"  image={image_height}x{image_width}, batch={train_batch_size}")
print(f"  aug: random_erase={random_erase}, color_jitter={color_jitter}")
print(f"  loss: margin={margin}, label_smooth={label_smooth}")

Output files in logs/mobilenet_v3_small-veri-lr3e-4:
  best_model.pth.tar                        20,447,819 bytes
  config.json                                    1,171 bytes
  eval_metrics.png                              55,150 bytes
  log_train.txt                                  7,093 bytes
  metrics.csv                                      447 bytes
  metrics.json                                     834 bytes
  model.pth.tar-1                           20,447,819 bytes
  plots/grad_norm_vs_epoch.png                  20,887 bytes
  plots/id_loss_vs_epoch.png                    19,769 bytes
  plots/loss_vs_epoch.png                       21,115 bytes
  plots/lr_vs_epoch.png                         29,437 bytes
  plots/triplet_loss_vs_epoch.png               20,202 bytes
  plots/val_map_vs_epoch.png                    20,090 bytes
  plots/val_minp_vs_epoch.png                   20,463 bytes
  plots/val_rank1_vs_epoch.png                  18,675 bytes
  retrieval_metrics.json        